In [22]:
from pathlib import Path
import pandas as pd
from preprocessing import clean_comments

In [23]:
DATA_PATH = Path('comments_data.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Tidak menemukan {DATA_PATH.resolve()}. Jalankan notebook dari folder proyek atau sesuaikan DATA_PATH.')

df = pd.read_csv(DATA_PATH)
print(f'Jumlah baris: {len(df):,}')
print(f'Kolom: {list(df.columns)}')
print(df.head().to_string(index=False))

Jumlah baris: 7,639
Kolom: ['post_url', 'post_id', 'account', 'posted_at', 'comment_id', 'text', 'commented_at', 'like_count', 'reply_count', 'topic_group', 'scrape_status', 'source_attempt']
                                post_url     post_id     account  posted_at        comment_id                                                text              commented_at  like_count  reply_count             topic_group scrape_status source_attempt
https://www.instagram.com/p/DeocT0SHONR/ DeocT0SHONR kumparancom        NaN 18024313715868092 cairrrrrr....caiiirrr kumparan angkat berita ini .. 2026-10-18T10:18:08+00:00         NaN          NaN survey_kepuasan_kinerja      complete      attempt_2
https://www.instagram.com/p/DeocT0SHONR/ DeocT0SHONR kumparancom        NaN 17973123492150905                             dibayar brp min? wkwkwk 2026-10-18T10:30:52+00:00         NaN          NaN survey_kepuasan_kinerja      complete      attempt_2
https://www.instagram.com/p/DeocT0SHONR/ DeocT0SHONR kumpa

In [24]:
if 'text' not in df.columns:
    raise KeyError("Dataset harus memiliki kolom 'text'")

text = df['text'].fillna('').astype(str)
audit = pd.Series({
    'total_baris': len(df),
    'teks_kosong_awal': int(text.str.strip().eq('').sum()),
    'teks_duplikat_di luar kemunculan pertama': int(text.duplicated().sum()),
    'teks_unik': int(text.nunique()),
    'komentar_panjang_0-3 karakter_sebelum_cleaning': int(text.str.len().le(3).sum()),
})
print(audit.to_frame('jumlah').to_string())
print(text.str.len().describe().to_frame('panjang_karakter').to_string())

                                                jumlah
total_baris                                       7639
teks_kosong_awal                                     0
teks_duplikat_di luar kemunculan pertama           742
teks_unik                                         6897
komentar_panjang_0-3 karakter_sebelum_cleaning     471
       panjang_karakter
count       7639.000000
mean          34.757691
std           49.407318
min            1.000000
25%           13.000000
50%           25.000000
75%           43.000000
max         1845.000000


In [25]:
cleaned_df, excluded_df = clean_comments(df, text_column='text', min_length=4)

print(f'Baris awal       : {len(df):,}')
print(f'Dipertahankan    : {len(cleaned_df):,}')
print(f'Dikeluarkan      : {len(excluded_df):,}')
print(f'Teks unik bersih : {cleaned_df["text_clean"].nunique():,}')

if not excluded_df.empty:
    print(excluded_df['exclusion_reason'].value_counts().rename_axis('alasan').to_frame('jumlah').to_string())
print(cleaned_df[['text', 'text_clean']].head(10).to_string(index=False))

Baris awal       : 7,639
Dipertahankan    : 6,873
Dikeluarkan      : 766
Teks unik bersih : 6,566
                      jumlah
alasan                      
empty_after_cleaning     683
short_text                83
                                                                                                             text                                                                                                     text_clean
                                                              cairrrrrr....caiiirrr kumparan angkat berita ini ..                                                            cairrrrrr....caiiirrr kumparan angkat berita ini ..
                                                                                          dibayar brp min? wkwkwk                                                                                        dibayar brp min? wkwkwk
                                                                              Yukkk suvei balik Tombol gak puas

In [27]:
cleaned_df = cleaned_df[['post_id', 'comment_id', 'text_clean']]

In [28]:
cleaned_df = cleaned_df.drop_duplicates(subset=['text_clean'], keep='first')

In [29]:
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = OUTPUT_DIR / 'comments_cleaned.csv'
excluded_path = OUTPUT_DIR / 'comments_excluded.csv'
cleaned_df.to_csv(cleaned_path, index=False)
excluded_df.to_csv(excluded_path, index=False)

print(f'Tersimpan: {cleaned_path.resolve()}')
print(f'Tersimpan: {excluded_path.resolve()}')

Tersimpan: /home/dzikri/Project/instagram-comment-preprocessing/outputs/comments_cleaned.csv
Tersimpan: /home/dzikri/Project/instagram-comment-preprocessing/outputs/comments_excluded.csv
